In [13]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def load_model(base_model_name, adapter_path):

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        device_map="auto"
    )

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    model = PeftModel.from_pretrained(base_model, adapter_path)

    return model, tokenizer

In [14]:
def gen_prompt(text):
    return f"Convert to structured JSON:\n{text}"

def edit_prompt(state, instruction):
    return f"""
Update JSON:

{json.dumps(state)}

Instruction:
{instruction}

Return JSON only.
"""

In [15]:
import re

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            return None
    return None

In [16]:
def generate(model, tokenizer, user_input):
    prompt = gen_prompt(user_input)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=200)

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return extract_json(text)


def edit(model, tokenizer, state, instruction):
    prompt = edit_prompt(state, instruction)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=200)

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return extract_json(text)

In [17]:
class JDSystem:

    def __init__(self):
        self.gen_model, self.gen_tok = load_model(
            "models/gemma-2b-it",
            "./models/gemma-2b-it-fine-tuned"
        )

        self.edit_model, self.edit_tok = load_model(
            "models/gemma-2b-it",
            "./models/gemma-2b-it-fine-tuned-edit"
        )

        self.state = None

    def run(self, user_input):

        if self.state is None:
            output = generate(self.gen_model, self.gen_tok, user_input)
            self.state = output
            mode = "generate"
        else:
            output = edit(self.edit_model, self.edit_tok, self.state, user_input)
            if output:
                self.state = output
            mode = "edit"

        return mode, self.state

In [18]:
import json
import datetime

LOG_FILE = "data/logs.jsonl"

def log_data(input_text, output, feedback=None):

    record = {
        "timestamp": str(datetime.datetime.now()),
        "input": input_text,
        "output": output,
        "feedback": feedback
    }

    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")

In [19]:
def get_feedback(output):

    print("\nModel Output:\n", json.dumps(output, indent=2))

    feedback = input("\nIs this correct? (y/n): ")

    if feedback == "y":
        return None
    else:
        corrected = input("\nEnter corrected JSON:\n")
        try:
            return json.loads(corrected)
        except:
            print("Invalid JSON")
            return None

In [20]:
def build_dataset():

    dataset = []

    with open("data/logs.jsonl") as f:
        for line in f:
            row = json.loads(line)

            if row["feedback"]:

                sample = {
                    "input": f"""
Existing JSON:
{row['output']}

Instruction:
Fix the output
""",
                    "output": json.dumps(row["feedback"])
                }

                dataset.append(sample)

    with open("data/dataset.json", "w") as f:
        json.dump(dataset, f, indent=2)

    print(f"Dataset built with {len(dataset)} samples")

In [21]:
import os

def should_retrain(threshold=50):

    if not os.path.exists("data/logs.jsonl"):
        return False

    count = sum(1 for _ in open("data/logs.jsonl"))

    return count >= threshold

In [22]:
def retrain():

    print("🔁 Retraining model...")

    # Load dataset
    with open("data/dataset.json") as f:
        data = json.load(f)

    # You plug your earlier fine-tuning code here

    print("✅ Training complete")

    # Save new version
    version = len(os.listdir("models/versions")) + 1

    save_path = f"models/versions/v{version}"

    model.save_pretrained(save_path)

    print(f"Saved new model: v{version}")

In [ ]:
system = JDSystem()

while True:

    user_input = input("\n>> ")

    if user_input == "exit":
        break

    mode, output = system.run(user_input)

    # Log
    feedback = get_feedback(output)

    log_data(user_input, output, feedback)

    # Build dataset
    build_dataset()

    # Retrain if enough data
    if should_retrain():
        retrain()